# TerraClimate Extraction: New Variables

This notebook extracts selected TerraClimate parameters for the EY Water Quality Challenge, following the method demonstrated in the TerraClimate Demonstration and Data Extraction notebooks. It uses the Microsoft Planetary Computer STAC API to access TerraClimate Zarr data.

## Variables Extracted (9)

We extract the following TerraClimate variables:

| Variable | Description |
|----------|-------------|
| `pet` | Reference Evapotranspiration (mm) |
| `ppt` | Precipitation accumulation (mm) |
| `tmax` | Maximum temperature (°C) |
| `tmin` | Minimum temperature (°C) |
| `soil` | Soil moisture at end of month (mm) |
| `aet` | Actual Evapotranspiration (mm) |
| `def` | Climate Water Deficit (mm) |
| `pdsi` | Palmer Drought Severity Index |
| `q` | Runoff (mm) |

**Excluded**: `srad, swe, vap, vpd, ws`, and `ppt_station_influence`.


## Step 1: Load Dependencies

We use `xarray` for Zarr/NetCDF, `pystac_client` and `planetary_computer` for STAC access, and `scipy.spatial.cKDTree` for nearest-neighbor mapping of climate values to sample locations.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xarray as xr
from scipy.spatial import cKDTree

import pystac_client
import planetary_computer as pc

from tqdm import tqdm
import os

## Step 2: Define Load, Filter, and Assignment Functions

These functions mirror the TerraClimate Demonstration and Data Extraction notebooks:

- **load_terraclimate_dataset()**: Opens the TerraClimate Zarr dataset from Microsoft Planetary Computer.
- **filterg(ds, var)**: Filters a given variable for 2011–2015 and the study region (Southern Africa bbox: lat -35.18 to -21.72, lon 14.97 to 32.79), returning a DataFrame.
- **assign_nearest_climate(sa_df, climate_df, var_name)**: Maps each sample (Latitude, Longitude, Sample Date) to the nearest TerraClimate grid point and assigns the value at the closest time stamp.

In [ ]:
def load_terraclimate_dataset():
    """Open TerraClimate Zarr dataset from Microsoft Planetary Computer."""
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )
    collection = catalog.get_collection("terraclimate")
    asset = collection.assets["zarr-abfs"]

    if "xarray:storage_options" in asset.extra_fields:
        ds = xr.open_zarr(
            asset.href,
            storage_options=asset.extra_fields["xarray:storage_options"],
            consolidated=True,
        )
    else:
        ds = xr.open_dataset(
            asset.href,
            **asset.extra_fields.get("xarray:open_kwargs", {}),
        )
    return ds

print("Done")

In [ ]:
def filterg(ds, var):
    """
    Filter TerraClimate variable for 2011–2015 and study region (Southern Africa).
    Returns a DataFrame with Latitude, Longitude, Sample Date, and the variable column.
    Auto-re-authenticates if signed Azure Storage URLs expire mid-loop.
    """
    ds_2011_2015 = ds[var].sel(time=slice("2011-01-01", "2015-12-31"))

    df_var_append = []
    for i in tqdm(range(len(ds_2011_2015.time)), desc=f"Filter {var}"):
        try:
            df_var = ds_2011_2015.isel(time=i).to_dataframe().reset_index()
        except Exception:
            # Signed URLs expired mid-loop — re-authenticate and retry this time step
            ds_fresh = load_terraclimate_dataset()
            ds_2011_2015 = ds_fresh[var].sel(time=slice("2011-01-01", "2015-12-31"))
            df_var = ds_2011_2015.isel(time=i).to_dataframe().reset_index()

        df_var_filter = df_var[
            (df_var["lat"] > -35.18) & (df_var["lat"] < -21.72)
            & (df_var["lon"] > 14.97)
            & (df_var["lon"] < 32.79)
        ]
        df_var_append.append(df_var_filter)

    df_var_final = pd.concat(df_var_append, ignore_index=True)
    df_var_final["time"] = df_var_final["time"].astype(str)
    col_mapping = {"lat": "Latitude", "lon": "Longitude", "time": "Sample Date"}
    df_var_final = df_var_final.rename(columns=col_mapping)
    return df_var_final

In [ ]:
def assign_nearest_climate(sa_df, climate_df, var_name):
    """
    Map each sample (Lat, Lon, Date) to the nearest TerraClimate grid point
    and assign the value at the closest time stamp.
    Returns a DataFrame with a single column: the variable values.
    """
    sa_coords = np.radians(sa_df[["Latitude", "Longitude"]].values)
    climate_coords = np.radians(climate_df[["Latitude", "Longitude"]].values)

    tree = cKDTree(climate_coords)
    dist, idx = tree.query(sa_coords, k=1)

    nearest_points = climate_df.iloc[idx].reset_index(drop=True)
    sa_df = sa_df.reset_index(drop=True)
    sa_df[["nearest_lat", "nearest_lon"]] = nearest_points[["Latitude", "Longitude"]]

    sa_dates = pd.to_datetime(sa_df["Sample Date"], dayfirst=True, errors="coerce")
    climate_df_local = climate_df.copy()
    climate_df_local["Sample Date"] = pd.to_datetime(
        climate_df_local["Sample Date"], dayfirst=True, errors="coerce"
    )

    climate_values = []
    for i in tqdm(range(len(sa_df)), desc=f"Map {var_name}"):
        sample_date = sa_dates.iloc[i]
        if pd.isna(sample_date):
            climate_values.append(np.nan)
            continue
        nearest_lat = sa_df.loc[i, "nearest_lat"]
        nearest_lon = sa_df.loc[i, "nearest_lon"]

        subset = climate_df_local[
            (climate_df_local["Latitude"] == nearest_lat)
            & (climate_df_local["Longitude"] == nearest_lon)
        ]

        if subset.empty:
            climate_values.append(np.nan)
            continue

        nearest_idx = (subset["Sample Date"] - sample_date).abs().idxmin()
        climate_values.append(subset.loc[nearest_idx, var_name])

    return pd.DataFrame({var_name: climate_values})

## Step 3: Define Variables and Load Sample Locations

We define the list of TerraClimate variables to extract. Sample locations are loaded from the training and validation CSV files.


In [ ]:
# TerraClimate variables to extract (excluding srad, swe, vap, vpd, ws)
VARIABLES_TO_EXTRACT = [
    "pet", "ppt", "tmax", "tmin", "soil", "aet", "def", "pdsi", "q"
]

# Load sample locations (same sources as TerraClimate_Data_Extraction_Notebook)
Water_Quality_df = pd.read_csv("water_quality_training_dataset.csv")
Validation_df = pd.read_csv("submission_template.csv")

# Keep only merge keys for extraction
train_samples = Water_Quality_df[["Latitude", "Longitude", "Sample Date"]].copy()
val_samples = Validation_df[["Latitude", "Longitude", "Sample Date"]].copy()

print(f"Training samples: {len(train_samples)}")
print(f"Validation samples: {len(val_samples)}")
print(f"Variables to extract: {VARIABLES_TO_EXTRACT}")


## Step 4: Load TerraClimate Dataset

We load the TerraClimate Zarr dataset from Microsoft Planetary Computer once. Each variable is then filtered and extracted in its own isolated cell in Step 5.


In [ ]:
print("Loading TerraClimate dataset (this may take a moment)...")
ds = load_terraclimate_dataset()
print("Dataset loaded. Each variable will be extracted in its own cell below.")


## Step 5: Extract Each Variable for Training and Validation

Each variable is processed in its **own isolated cell** to avoid runtime errors during data extraction. For each variable we: (1) filter the region and time, (2) assign nearest climate values to training and validation samples, (3) store in the output DataFrames.

**Note**: Run each cell sequentially. Extraction is time-consuming (each variable iterates over ~60 months for filtering and ~9k+ samples for mapping).


In [ ]:
# Initialize output DataFrames with merge keys; reset index for positional alignment
train_out = train_samples.reset_index(drop=True).copy()
val_out = val_samples.reset_index(drop=True).copy()

In [ ]:
# --- pet (Reference Evapotranspiration) ---
var = "pet"
print(f"Extracting {var.upper()}...")
ds = load_terraclimate_dataset()  # Refresh auth (signed URLs expire ~1hr)
tc_df = filterg(ds, var)
train_var = assign_nearest_climate(train_samples, tc_df, var)
val_var = assign_nearest_climate(val_samples, tc_df, var)
train_arr = train_var[var].astype(float).to_numpy()
val_arr = val_var[var].astype(float).to_numpy()
assert len(train_arr) == len(train_out), f"Length mismatch for {var}"
assert len(val_arr) == len(val_out), f"Length mismatch for {var}"
train_out[var] = train_arr
val_out[var] = val_arr
print(f"Done: {var}")


In [ ]:
train_var.head()

In [ ]:
# --- ppt (Precipitation accumulation) ---
var = "ppt"
print(f"Extracting {var.upper()}...")
ds = load_terraclimate_dataset()  # Refresh auth (signed URLs expire ~1hr)
tc_df = filterg(ds, var)
train_var = assign_nearest_climate(train_samples, tc_df, var)
val_var = assign_nearest_climate(val_samples, tc_df, var)
train_arr = train_var[var].astype(float).to_numpy()
val_arr = val_var[var].astype(float).to_numpy()
assert len(train_arr) == len(train_out), f"Length mismatch for {var}"
assert len(val_arr) == len(val_out), f"Length mismatch for {var}"
train_out[var] = train_arr
val_out[var] = val_arr
print(f"Done: {var}")


In [ ]:
# --- tmax (Maximum temperature) ---
var = "tmax"
print(f"Extracting {var.upper()}...")
ds = load_terraclimate_dataset()  # Refresh auth (signed URLs expire ~1hr)
tc_df = filterg(ds, var)
train_var = assign_nearest_climate(train_samples, tc_df, var)
val_var = assign_nearest_climate(val_samples, tc_df, var)
train_arr = train_var[var].astype(float).to_numpy()
val_arr = val_var[var].astype(float).to_numpy()
assert len(train_arr) == len(train_out), f"Length mismatch for {var}"
assert len(val_arr) == len(val_out), f"Length mismatch for {var}"
train_out[var] = train_arr
val_out[var] = val_arr
print(f"Done: {var}")


In [ ]:
# --- tmin (Minimum temperature) ---
var = "tmin"
print(f"Extracting {var.upper()}...")
ds = load_terraclimate_dataset()  # Refresh auth (signed URLs expire ~1hr)
tc_df = filterg(ds, var)
train_var = assign_nearest_climate(train_samples, tc_df, var)
val_var = assign_nearest_climate(val_samples, tc_df, var)
train_arr = train_var[var].astype(float).to_numpy()
val_arr = val_var[var].astype(float).to_numpy()
assert len(train_arr) == len(train_out), f"Length mismatch for {var}"
assert len(val_arr) == len(val_out), f"Length mismatch for {var}"
train_out[var] = train_arr
val_out[var] = val_arr
print(f"Done: {var}")


In [ ]:
# --- soil (Soil moisture at end of month) ---
var = "soil"
print(f"Extracting {var.upper()}...")
ds = load_terraclimate_dataset()  # Refresh auth (signed URLs expire ~1hr)
tc_df = filterg(ds, var)
train_var = assign_nearest_climate(train_samples, tc_df, var)
val_var = assign_nearest_climate(val_samples, tc_df, var)
train_arr = train_var[var].astype(float).to_numpy()
val_arr = val_var[var].astype(float).to_numpy()
assert len(train_arr) == len(train_out), f"Length mismatch for {var}"
assert len(val_arr) == len(val_out), f"Length mismatch for {var}"
train_out[var] = train_arr
val_out[var] = val_arr
print(f"Done: {var}")


In [ ]:
# --- aet (Actual Evapotranspiration) ---
var = "aet"
print(f"Extracting {var.upper()}...")
ds = load_terraclimate_dataset()  # Refresh auth (signed URLs expire ~1hr)
tc_df = filterg(ds, var)
train_var = assign_nearest_climate(train_samples, tc_df, var)
val_var = assign_nearest_climate(val_samples, tc_df, var)
train_arr = train_var[var].astype(float).to_numpy()
val_arr = val_var[var].astype(float).to_numpy()
assert len(train_arr) == len(train_out), f"Length mismatch for {var}"
assert len(val_arr) == len(val_out), f"Length mismatch for {var}"
train_out[var] = train_arr
val_out[var] = val_arr
print(f"Done: {var}")


In [ ]:
# --- def (Climate Water Deficit) ---
var = "def"
print(f"Extracting {var.upper()}...")
ds = load_terraclimate_dataset()  # Refresh auth (signed URLs expire ~1hr)
tc_df = filterg(ds, var)
train_var = assign_nearest_climate(train_samples, tc_df, var)
val_var = assign_nearest_climate(val_samples, tc_df, var)
train_arr = train_var[var].astype(float).to_numpy()
val_arr = val_var[var].astype(float).to_numpy()
assert len(train_arr) == len(train_out), f"Length mismatch for {var}"
assert len(val_arr) == len(val_out), f"Length mismatch for {var}"
train_out[var] = train_arr
val_out[var] = val_arr
print(f"Done: {var}")


In [ ]:
# --- pdsi (Palmer Drought Severity Index) ---
var = "pdsi"
print(f"Extracting {var.upper()}...")
ds = load_terraclimate_dataset()  # Refresh auth (signed URLs expire ~1hr)
tc_df = filterg(ds, var)
train_var = assign_nearest_climate(train_samples, tc_df, var)
val_var = assign_nearest_climate(val_samples, tc_df, var)
train_arr = train_var[var].astype(float).to_numpy()
val_arr = val_var[var].astype(float).to_numpy()
assert len(train_arr) == len(train_out), f"Length mismatch for {var}"
assert len(val_arr) == len(val_out), f"Length mismatch for {var}"
train_out[var] = train_arr
val_out[var] = val_arr
print(f"Done: {var}")


In [ ]:
# --- q (Runoff) ---
var = "q"
print(f"Extracting {var.upper()}...")
ds = load_terraclimate_dataset()  # Refresh auth (signed URLs expire ~1hr)
tc_df = filterg(ds, var)
train_var = assign_nearest_climate(train_samples, tc_df, var)
val_var = assign_nearest_climate(val_samples, tc_df, var)
train_arr = train_var[var].astype(float).to_numpy()
val_arr = val_var[var].astype(float).to_numpy()
assert len(train_arr) == len(train_out), f"Length mismatch for {var}"
assert len(val_arr) == len(val_out), f"Length mismatch for {var}"
train_out[var] = train_arr
val_out[var] = val_arr
print(f"Done: {var}")


## Step 6: Save Output Files

We save the training and validation feature files. These can be merged with Landsat, spatial, and temporal features in the combined modeling pipeline.

In [ ]:
out_train = "terraclimate_features_training_new_variables.csv"
out_val = "terraclimate_features_validation_new_variables.csv"

train_out.to_csv(out_train, index=False)
val_out.to_csv(out_val, index=False)

print(f"Saved {out_train} ({len(train_out)} rows, {len(train_out.columns)} columns)")
print(f"Saved {out_val} ({len(val_out)} rows, {len(val_out.columns)} columns)")
print("\nColumns:", list(train_out.columns))
train_out.head(10)